In [ ]:
%pip install pandas-market-calendars

In [1]:
import lseg.data as ld
import pandas as pd
import numpy as np
import pandas_market_calendars as mcal
import pytz

from datetime import date, time, timedelta, datetime
from functools import reduce

ld.open_session()

<lseg.data.session.Definition object at 0x7f303e47e550 {name='codebook'}>

In [2]:
# HELPER FUNCTIONS
def get_trading_days(n, end_date=date.today()):
    nyse = mcal.get_calendar('NYSE')
    trading_days = nyse.valid_days(start_date=end_date - timedelta(days=n * 2),  # Start with a wider range
                                    end_date=end_date)
    
    if len(trading_days) >= n:
        return trading_days[-n:]  #take last n trading days
    else:
        #if not enough days, extend the range.
        extended_start_date = end_date - timedelta(days=n*3)
        trading_days = nyse.valid_days(start_date=extended_start_date,  
                                    end_date=end_date)
        return trading_days[-n:]
    
# # SET DATE RANGE
# Use pytz.all_timezones to find a full list of available timezones
tz_NY = pytz.timezone('America/New_York')

# Get the current datetime for the specified timezone and extract the date
# today = datetime.now(tz_NY).date()
today = date.today()
past = get_trading_days(5, today)[0].date()

# MORE HELPER FUNCTIONS
def merge_pivoted(left, right):
    return pd.merge(left, right, left_index=True, right_index=True, how='outer')

def pct_change(val1, val2):
    return (val2 - val1)/val1

def get_changes(df):
    out = df.copy()
    val1s = list(df.iloc[0])
    
    for i in range(len(df.columns)):
        val2s = list(df.iloc[:, i])
        changes = [pct_change(val1s[i], x) for x in val2s]
        
        out[df.columns[i]] = changes
        
    return out

def format_currency(value):
    return '${:,.2f}'.format(value)

def format_currency_round(value):
    return '${:,.0f}'.format(value)

def format_percentage(val):
    return '{:.2f}%'.format(val * 100)

def format_int(val):
    return '{:,}'.format(val)

def latest_sharesOutstanding(ids, first=past, last=today):
    d = {}

    sharesOutstanding = ld.get_history(
                universe=ids,
                fields=['TR.F.ComShrOutsTot'],
                interval="10min", # TEST CHANGING THIS...
                start = str(first),
                end = str(last)
                )
        
    for c in sharesOutstanding.columns:
        # Filter for numbers
        x = [t for t in sharesOutstanding[c] if np.issubdtype(type(t), np.number) and pd.notna(t)]
        
        if len(x) > 0:
            d[c] = int(x[-1])
        else:
            # Handle the missing data! 
            # You can set it to 0, or log which ID failed
            print(f"Warning: No shares data found for {c}")
            d[c] = 0 
    
    return d

def intraday_mktCap(df):
    
    d = {}
    # ids = df.columns.tolist()
    ids = [i for i in df.columns if i.lower() not in ['date', 'timestamp']]
    # print(len(ids))
    shares = latest_sharesOutstanding(ids)
    for i in ids:
        d[i] = shares[i] * df[i]
        
    return d

def get_avgPrice(df_):
    # 1. Get Market Cap and ensure it's numeric
    mkCap = intraday_mktCap(df_)
    df = pd.DataFrame(mkCap).apply(pd.to_numeric, errors='coerce')

    # 2. Calculate Weights (Vectorized)
    # total_daily_mktCap is a Series where each value is the sum of the row
    total_daily_mktCap = df.sum(axis=1)
    
    # Divide the entire DataFrame by the row sums to get weights
    weightDf = df.div(total_daily_mktCap, axis=0)

    # 3. Prepare Prices
    # Filter 'concatted' to only include numeric columns that match weightDf
    prices = concatted.select_dtypes(include=['number']).reset_index(drop=True)
    
    # 4. Calculate Weighted Average (Vectorized)
    # This multiplies matching columns automatically
    weightAvg = prices.mul(weightDf.reset_index(drop=True))

    # 5. Final Index Calculation
    weightAvg['CyberIndex'] = weightAvg.sum(axis=1)
    weightAvg.index = df.index # Restore the original index
    
    return weightAvg

# 3. Apply the custom Excel formatting (mm/dd/yyyy hh:mm:ss a/p)
def format_to_excel_ap(dt):
    # Standard format with AM/PM
    base = dt.strftime('%m/%d/%Y %I:%M:%S')
    # Convert 'AM' to 'a' and 'PM' to 'p'
    ap = 'a' if dt.hour < 12 else 'p'
    return f"{base} {ap}"

# company universe
ids = ['ALLT.OQ','INTZ.OQ','MITK.OQ','OSPN.OQ','CISO.OQ','CSCO.OQ',
       'PANW.OQ','CRWD.OQ','FTNT.OQ','ZS.OQ','CHKP.OQ','OKTA.OQ',
       'FFIV.OQ','AKAM.OQ','SAIL.OQ','VRNS.OQ','QLYS.OQ',
       'TENB.OQ','NTCT.OQ','RDWR.OQ','RPD.OQ','TLS.OQ','HUBC.OQ',
       'CYCU.OQ','NET.N','LDOS.N','RBRK.N','S.N','LUMN.N',
       'ATEN.N', 'CVLT.OQ']

# GET DATA
top = ld.get_data(
            universe=ids,
            fields=['TR.CommonName','TR.TickerSymbol','CF_Date','CF_Close', 
                    'CF_Volume', 'TR.CompanyMarketCap'], 
            parameters={
                'SDate': str(today),
                'EDate': str(today)}
)

ordered_top = top.sort_values(by='Company Market Cap', ascending=False).reset_index(drop=True)

top_20_table = ordered_top.head(20)
# print(f"date_range: {past} to {today}")

# 10 MINUTE PRICE INTERVALS UP TO TODAY
hist = ld.get_history(
            universe=[i for i in top_20_table.Instrument],
            fields=['TRDPRC_1'],
            interval="10min", # TEST CHANGING THIS...
            start = str(past), # can this be generalized?
            end = f"{today} 00:00:00"
            )
hist = hist.reset_index()
hist = hist.ffill()
hist = hist.bfill()
hist.rename(columns={'Timestamp': 'date'}, inplace=True)

# INTRADAY PRICES FOR TODAY
intra = ld.get_history(
            universe=[i for i in top_20_table.Instrument],
            fields=['TRDPRC_1'],
            interval="10min", # TEST CHANGING THIS...
            start = str(today), # can this be generalized?
            # end = str(today)
            )

intra = intra.reset_index()
intra = intra.ffill()
intra = intra.bfill()
intra.rename(columns={'Timestamp': 'date'}, inplace=True)

# concatted is prices for the top 20 stocks in 10 minute intervals
concatted = pd.concat([hist, intra], ignore_index=False)
concatted['date'] = pd.to_datetime(concatted['date'])
concatted = concatted.set_index('date').sort_index()

# Incoming data is UTC, then convert to New York Time
if concatted.index.tz is None:
    concatted.index = concatted.index.tz_localize('UTC')
concatted.index = concatted.index.tz_convert('America/New_York')

# calculated intraday marketcap and average weighted price for group of 20 stocks
weightedAvg = get_avgPrice(concatted)
concatted['CyberIndex'] = weightedAvg['CyberIndex']

# Filter the DataFrame by time
start_time = time(9, 30)  # 9:30 AM
end_time = time(16, 0)    # 4:00 PM
time_mask = (concatted.index.time >= start_time) & (concatted.index.time <= end_time)
concatted = concatted[time_mask]

# calculate percent changes from start date
changes = get_changes(concatted)

# Filter the DataFrame by time
start_time = time(9, 30)  # 9:30 AM
end_time = time(16, 0)    # 4:00 PM
time_mask = (changes.index.time >= start_time) & (changes.index.time <= end_time)

time_changes = changes[time_mask]
time_changes = time_changes.reset_index()

time_changes.columns = [
    'Date' if str(col).lower() in ['date', 'timestamp', "('trdprc_1', 'date')"] 
    else (col[1] if isinstance(col, tuple) else col) 
    for col in time_changes.columns
]

# Ensure 'Date' is datetime, then apply the custom string format
time_changes['Date'] = pd.to_datetime(time_changes['Date']).apply(format_to_excel_ap)

# generate display table 
outTable = top_20_table[['Instrument', 'Company Common Name', 'Ticker Symbol',
                         'CF_DATE', 'CF_VOLUME', 'Company Market Cap']]

concat_timeMask = (concatted.index.time >= start_time) & (concatted.index.time <= end_time)
concatted = concatted[concat_timeMask]

# Slicing with [-1:] keeps it as a DataFrame instead of a Series
last_prices = concatted.iloc[-1:].T.reset_index()
# The columns in last_prices will be 'index' (or 'Instrument') and the timestamp
last_prices.columns = ['Instrument', 'Price Close']

# Merge with outTable
outTable = pd.merge(outTable, last_prices, on='Instrument', how='left')
    
# calculate weekly change
wklyChange = time_changes.set_index('Date').iloc[-1]
outTable['Weekly Change'] = outTable['Instrument'].map(wklyChange)

new_names = {'Company Common Name': 'Company',
             'Ticker Symbol': 'Ticker',
             'CF_DATE': 'Date',
             'Price Close': 'Price Close',
             'Weekly Change': 'Weekly Change',
             'CF_VOLUME': 'Volume',
             'Company Market Cap': 'Market Cap'
            }

# map for LSEG names to common company names 
com_name = ld.get_data(
            universe=ids,
            fields=['TR.CommonName'], 
)

com_name_map = {inst: comname.replace(" Inc", "").replace(" Ltd", "") for inst, comname in zip(com_name['Instrument'], com_name['Company Common Name'])}
time_changes = time_changes.rename(columns=com_name_map)

# format table
top_20_out = outTable[[k for k in new_names.keys()]].copy()
top_20_out.rename(columns=new_names, inplace=True)
top_20_out['Market Cap'] = top_20_out['Market Cap'].apply(format_currency_round)
top_20_out['Weekly Change'] = top_20_out['Weekly Change'].apply(format_percentage)
top_20_out['Volume'] = top_20_out['Volume'].apply(format_int)
top_20_out['Price Close'] = top_20_out['Price Close'].apply(format_currency)
top_20_out['Company'] = top_20_out['Company'].str.replace(r' ltd| inc', '', case=False, regex=True).str.strip()
top_20_out.index += 1

print(f'CyberIndex data pull complete for dates {past} to {today}')

CyberIndex data pull complete for dates 2026-03-03 to 2026-03-09


In [3]:
# CHARTING DATA
time_changes

,Date,Cisco Systems,Palo Alto Networks,CrowdStrike Holdings,Cloudflare,Fortinet,Zscaler,Leidos Holdings,Check Point Software Technologies,F5,...,Rubrik,SailPoint,Lumen Technologies,SentinelOne,Commvault Systems,Qualys,Varonis Systems,Tenable Holdings,Netscout Systems,CyberIndex
0,03/03/2026 09:30:00 a,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,03/03/2026 09:40:00 a,-0.001144,0.001261,-0.001088,0.003210,0.000063,0.010050,-0.004621,0.004350,0.002225,...,-0.002166,-0.002896,-0.006024,-0.004644,0.010655,0.002713,-0.000443,0.002621,-0.004541,0.000414
2,03/03/2026 09:50:00 a,-0.004957,0.005841,-0.003672,0.011176,0.003964,0.007384,0.004339,0.001169,0.006275,...,0.006499,-0.001448,-0.003012,-0.002322,0.010776,-0.010586,0.000887,0.004717,0.000000,0.000927
3,03/03/2026 10:00:00 a,-0.005085,-0.005045,-0.006648,0.005387,0.001762,-0.009024,0.002254,-0.002402,0.001350,...,-0.000591,-0.003621,0.004518,-0.010836,0.015014,-0.010160,-0.009309,-0.004193,-0.006636,-0.003918
4,03/03/2026 10:10:00 a,-0.009915,-0.000962,-0.008261,0.000115,0.002769,-0.006290,0.010595,0.005389,-0.007278,...,-0.003348,-0.003983,0.001506,-0.005418,0.021915,-0.009947,-0.007535,0.004193,-0.005589,-0.004929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,03/06/2026 03:20:00 p,-0.000636,0.090740,0.128950,0.118696,0.051472,0.118856,0.005213,0.076808,0.031739,...,0.122292,0.111513,-0.009036,0.092105,0.055091,0.039204,0.072695,0.096436,0.041216,0.106713
156,03/06/2026 03:30:00 p,-0.002352,0.094059,0.125056,0.114454,0.051976,0.120086,0.001296,0.076484,0.035533,...,0.119732,0.110065,-0.008283,0.093266,0.051580,0.043353,0.072695,0.099057,0.039120,0.104541
157,03/06/2026 03:40:00 p,-0.000508,0.095055,0.123731,0.117033,0.051598,0.120667,0.003607,0.073789,0.037211,...,0.127215,0.111151,-0.006777,0.096749,0.049401,0.042928,0.071587,0.100367,0.039469,0.104332
158,03/06/2026 03:50:00 p,-0.000381,0.095453,0.125043,0.118982,0.053108,0.121693,0.002705,0.072458,0.044216,...,0.126231,0.113324,-0.001506,0.099071,0.053881,0.048886,0.080674,0.103249,0.040866,0.105676


In [4]:
# TABLE DATA
top_20_out

,Company,Ticker,Date,Price Close,Weekly Change,Volume,Market Cap
1,Cisco Systems,CSCO,2026-03-06,$78.58,-0.11%,"30,557","$310,619,588,823"
2,Palo Alto Networks,PANW,2026-03-06,$164.95,9.49%,"12,243","$134,680,800,000"
3,CrowdStrike Holdings,CRWD,2026-03-06,$428.97,12.50%,"5,532","$108,797,908,469"
4,Cloudflare,NET,2026-03-06,$195.19,11.87%,<NA>,"$67,689,331,108"
5,Fortinet,FTNT,2026-03-06,$83.40,4.96%,"4,289","$61,909,406,190"
6,Zscaler,ZS,2026-03-06,$163.76,11.96%,"3,810","$26,379,322,734"
7,Leidos Holdings,LDOS,2026-03-06,$177.89,0.25%,<NA>,"$22,202,138,871"
8,Check Point Software Technologies,CHKP,2026-03-06,$165.22,7.27%,189,"$17,737,147,169"
9,F5,FFIV,2026-03-06,$286.22,4.42%,<NA>,"$16,176,946,318"
10,Akamai Technologies,AKAM,2026-03-06,$99.90,4.00%,45,"$14,472,873,707"


In [5]:
# SAVE DATA 
# SET 'save' to 'TRUE' to save charting and table data to file list
# download files to local machine 
# file download names will print below cell
save = True

changes = f'cyberIndex_ChartingData_{today}.csv'
changes = changes.replace('-', '_')


table = f'cyberIndex_Table_{today}.csv'
table = table.replace('-', '_')

if save:
    time_changes.to_csv(changes, float_format='%.6f', index=False)
    top_20_out.to_csv(table, index=False)

print(f'Download is {save}')
print(changes)
print(table)

Download is True
cyberIndex_ChartingData_2026_03_09.csv
cyberIndex_Table_2026_03_09.csv
